# **Imports** 

> Add blockquote



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

from transformers import BertTokenizerFast
from datasets import load_dataset

import numpy as np
import matplotlib.pyplot as plt
from torchmetrics import Accuracy

from tqdm import tqdm

C:\Users\PartoFix.com\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Dataset**

### laod Dataset

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")

In [ ]:
dataset.save_to_disk("D:/Demis2/ag_news_dataset")

# **Preprocessing**

In [ ]:
train_valid = dataset["train"].train_test_split(test_size=0.2,seed=42)       

In [7]:
train_valid.shape

{'train': (96000, 2), 'test': (24000, 2)}

In [16]:
train_dataset = train_valid["train"]
valid_dataset = train_valid["test"]
test_dataset = dataset["test"]

In [11]:
train_dataset.shape

(96000, 2)

### Tokenization


In [12]:
from transformers import BertTokenizerFast


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [13]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

c:\Users\microsoft\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\microsoft\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [15]:
tokenizer("demis is the Best")

{'input_ids': [101, 27668, 2015, 2003, 1996, 2190, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [14]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
sent = ('demis is the best', 'its students are awsome')

In [ ]:
tokenizer.convert_ids_to_tokens(101)


'[CLS]'

In [ ]:
tokenizer.convert_ids_to_tokens(102)

'[SEP]'

In [ ]:
tokenizer.convert_tokens_to_ids('Demis')

100

In [ ]:
tokenizer.convert_tokens_to_ids('[ukn]')

100

### Maapping

In [17]:
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


Map: 100%|██████████| 7600/7600 [00:00<00:00, 10192.96 examples/s]


In [18]:
train_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 96000
})

In [ ]:
#root_dir = r'C:\Teaching_DL\NLP'

In [ ]:
!pip install -U datasets transformers
from datasets import load_dataset

  Attempting uninstall: datasets
    Found existing installation: datasets 4.4.1
    Uninstalling datasets-4.4.1:
      Successfully uninstalled datasets-4.4.1



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
dataset = load_dataset("ag_news")

In [ ]:
train_set = dataset['train']
test_set = dataset['test']

In [ ]:
train_set.shape

(120000, 2)

In [ ]:
train_set.column_names

['text', 'label']

In [ ]:
test_set.shape

(7600, 2)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=10
    )

In [ ]:
train_datasets = train_set.map(tokenize, batched=True)
test_datasets = test_set.map(tokenize, batched=True)

In [ ]:
train_datasets[5]

{'text': 'Stocks End Up, But Near Year Lows (Reuters) Reuters - Stocks ended slightly higher on Friday\\but stayed near lows for the year as oil prices surged past  #36;46\\a barrel, offsetting a positive outlook from computer maker\\Dell Inc. (DELL.O)',
 'label': 2,
 'input_ids': [101, 15768, 2203, 2039, 1010, 2021, 2379, 2095, 2659, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
train_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# **DataLoader**

In [19]:
BATCH_SIZE = 32
train_loader = DataLoader(train_datasets, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

NameError: name 'DataLoader' is not defined

In [ ]:
x = next(iter(train_loader))

In [ ]:
input_ids = x['input_ids'].long()

In [ ]:
input_ids

In [ ]:
input_ids.dtype

AttributeError: 'list' object has no attribute 'dtype'

In [ ]:
print(input_ids.dtype)

torch.int64


In [ ]:
input_ids.shape

torch.Size([32, 10])

In [ ]:
for iter in train_loader:
  print(iter['input_ids'].shape)
  break

torch.Size([32, 10])


# **Model**

Pre_Trained Model

In [ ]:
import torch.nn as nn

In [ ]:
vocab_size = tokenizer.vocab_size

In [ ]:
embedding = nn.Embedding(30522, 128)

In [ ]:
embd = embedding(input_ids)

In [ ]:
embd.shape

torch.Size([32, 10, 128])

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, RNN, vocab_size, input_size, hidden_size, num_layers, bidirectional, num_cls):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, input_size)
        self.rnn = RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), num_cls)

    def forward(self, x):
        x = self.embedding(x)
        #x = x.permute(1, 0, 2)
        outputs, _ = self.rnn(x)
        y = self.fc(outputs)
        y = y.mean(dim=0)
        return y

In [ ]:
model = RNNModel(nn.RNN, vocab_size, 32, 128, 1, True, 4)

In [ ]:
model(input_ids)

tensor([[ 0.0665,  0.0401, -0.2279,  0.1454],
        [ 0.1110,  0.0678, -0.0149,  0.0508],
        [ 0.0532,  0.0973,  0.0375, -0.0271],
        [ 0.0645, -0.0077, -0.0103,  0.0017],
        [ 0.0538,  0.0756,  0.0249, -0.0252],
        [ 0.0699,  0.0908,  0.0356, -0.0057],
        [ 0.0674,  0.0117,  0.0671, -0.0458],
        [ 0.0367,  0.0627,  0.0165,  0.0383],
        [ 0.0307,  0.1060,  0.0194,  0.0285],
        [-0.0505, -0.0705,  0.2397, -0.1538]], grad_fn=<MeanBackward1>)

In [ ]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:

num_cls = 4
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
def train_one_epoch(model, train_loader, loss_fn, optimizer, epoch=None):
  model.train()
  loss_train = AverageMeter()
  acc_train = Accuracy().to(device)
  with tqdm(train_loader, unit="batch") as tepoch:
    for inputs, targets in tepoch:
      if epoch is not None:
        tepoch.set_description(f"Epoch {epoch}")
      inputs = inputs.to(device)
      targets = targets.to(device)

      outputs = model(inputs)

      loss = loss_fn(outputs, targets)

      loss.backward()

      optimizer.step()
      optimizer.zero_grad()

      loss_train.update(loss.item())
      acc_train(outputs, targets.int())
      tepoch.set_postfix(loss=loss_train.avg,
                         accuracy=100.*acc_train.compute().item())
  return model, loss_train.avg, acc_train.compute().item()

In [ ]:
def validation(model, test_loader, loss_fn):
  model.eval()
  with torch.no_grad():
    loss_valid = AverageMeter()
    acc_valid = Accuracy().to(device)
    for i, (inputs, targets) in enumerate(test_loader):
      inputs = inputs.to(device)
      targets = targets.to(device)

      outputs = model(inputs)
      loss = loss_fn(outputs, targets)

      loss_valid.update(loss.item())
      acc_valid(outputs, targets.int())
  return loss_valid.avg, acc_valid.compute().item()

In [ ]:
model = RNNModel(nn.RNN, vocab_size=vocab_size, input_size=32, hidden_size=64, num_layers=1, bidirectional=False, num_cls=num_cls).to(device)
loss_fn = nn.CrossEntropyLoss()

batch = next(iter(train_loader))

x_batch = batch['input_ids'].to(device)
y_batch = batch['label'].to(device)

outputs = model(x_batch)

# Print the shape of the outputs
print(f"outputs shape: {outputs.shape}")
outputs = model(x_batch.to(device))
outputs = outputs[:, -1, :]
loss = loss_fn(outputs, y_batch.to(device))
print(loss)

outputs shape: torch.Size([10, 4])


IndexError: too many indices for tensor of dimension 2